# Chapter 12: Training Your Model

> "I have not failed. I've just found 10,000 ways that won't work."
> — **Thomas Edison**, Inventor

---

## What You'll Learn

- How next-token prediction teaches models to write
- The 5-step training recipe that powers all neural networks
- Why loss curves reveal if your model is learning
- How to prevent overfitting and know when to stop
- Saving checkpoints so you never lose progress
- The thrill of watching your model transform from gibberish to coherence

---

## Setup

First, let's install required packages:

In [ ]:
# Install required packages (pins match the book's tested environment,
# Appendix A; torch/transformers are imported directly in the next cell)
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== IMPORTS =====
import math
import urllib.request
from functools import partial
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ===== REPRODUCIBILITY =====
import random
import numpy as np

def set_seed(seed=42):
    """Seed every random source we use."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)

## 1. Model Components from Chapters 10-11

First, let's bring in the MiniGPT model we built in previous chapters.

In [ ]:
# ===== MULTI-HEAD ATTENTION (from Chapter 10) =====
class MultiHeadAttention(nn.Module):
    """
    Efficient multi-head attention (batches all heads together).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Combined QKV projection (3x more efficient than separate!)
        # Why 3 * d_model? Because we project to Q, K, V simultaneously
        # bias=True matches GPT-2, so we can load its pretrained biases later
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout on attention weights
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Step 1: Project to Q, K, V (all at once!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Step 2: Split into Q, K, V and reshape for multi-head =====
        # Reshape to (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Permute to (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Split into Q, K, V: each is (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Step 3: Scaled dot-product attention (batched over heads) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Apply causal mask if provided
        if mask is not None:
            # Expand mask for heads: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Step 4: Concatenate heads =====
        # Transpose to (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Reshape to (batch, seq, d_model), this concatenates heads
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Step 5: Final projection =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Test it

print("MultiHeadAttention defined!")

In [ ]:
# ===== FEEDFORWARD NETWORK (from Chapter 10) =====
class FeedForward(nn.Module):
    """
    Position-wise feedforward network.
    Applied to each position independently (same weights for all positions).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model dimension (768 for GPT-2 small)
            d_ff: Feedforward dimension (typically 4 * d_model = 3072)
            dropout: Dropout probability
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), expand
        x = F.gelu(x, approximate="tanh")  # Non-linearity (GPT-2's GELU variant)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), project back
        return x


# Test it

print("FeedForward defined!")

In [ ]:
# ===== TRANSFORMER BLOCK (from Chapter 10) =====
class TransformerBlock(nn.Module):
    """
    Complete Transformer block with multi-head attention, feedforward,
    residuals, and layer normalization (pre-norm style like GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model dimension (768)
            num_heads: Number of attention heads (12)
            d_ff: Feedforward dimension (3072 = 4 * d_model)
            dropout: Dropout probability
        """
        super().__init__()

        # Layer normalization (before each sub-layer)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Multi-head attention
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Feedforward network
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (applied after each sub-layer)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Input embeddings (batch, seq, d_model)
            mask: Causal mask (seq, seq) or (1, 1, seq, seq)

        Returns:
            x: Output embeddings (batch, seq, d_model)
            attn_weights: Attention weights (batch, heads, seq, seq)
        """
        # ===== Multi-Head Attention with Residual =====
        # Pre-norm: Normalize BEFORE attention
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Residual connection

        # ===== Feedforward with Residual =====
        # Pre-norm: Normalize BEFORE FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Residual connection

        return x, attn_weights


# Test a complete Transformer block

print("TransformerBlock defined!")

In [ ]:
# ===== GPT CONFIG (from Chapter 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Configuration for MiniGPT model."""
    vocab_size: int = 50257      # GPT-2 vocabulary size
    max_seq_len: int = 1024      # Maximum context length
    embed_dim: int = 768         # Embedding dimension (d_model)
    num_heads: int = 12          # Number of attention heads
    num_layers: int = 12         # Number of Transformer blocks
    d_ff: int = 3072             # Feedforward hidden dimension (4 * embed_dim)
    dropout: float = 0.1         # Dropout probability

    def __post_init__(self):
        """Validate configuration after initialization."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) must divide by " \
            f"num_heads ({self.num_heads})"


In [ ]:
# ===== MINIGPT MODEL (from Chapter 11) =====
class MiniGPT(nn.Module):
    """
    A minimal GPT-style language model.
    Combines embeddings, Transformer blocks, and language modeling head.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token and Position Embeddings (Chapter 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blocks (Chapter 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Final LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Language Modeling Head =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Weight Tying =====
        # Share weights between embedding and output projection
        self.lm_head.weight = self.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize weights with small random values."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Forward pass through the model.

        Args:
            token_ids: Input token IDs (batch, seq)
            return_attention: Whether to return attention weights

        Returns:
            logits: Vocabulary scores (batch, seq, vocab_size)
            attention_weights: Optional list of attention weights per block
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Embeddings =====
        # Token embeddings
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Position embeddings
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Combine and apply dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Create Causal Mask =====
        # Lower triangular: each position can only attend to previous positions
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Pass Through Transformer Blocks =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Final LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Language Modeling Head =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Generate tokens continuing input_ids (prompt included in output).

        Args:
            input_ids: Prompt token IDs (batch, seq)
            max_new_tokens: How many new tokens to generate
            temperature: Sampling temperature (only used when do_sample=True)
            do_sample: Sample from the distribution; False = greedy argmax
            eos_token_id: Stop early if every sequence generates this token

        Returns:
            Token IDs (batch, seq + new tokens)
        """
        self.eval()

        # Position embeddings only cover max_seq_len positions
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} tokens) + max_new_tokens "
                f"({max_new_tokens}) exceeds the model's {limit}-token "
                "context. Shorten the prompt or lower max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


## 2. Download Dataset

We'll use TinyShakespeare - small enough to train in minutes.

In [ ]:
# Download TinyShakespeare
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "shakespeare.txt")

with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Dataset size: {len(text):,} characters")
print(f"\nSample:\n{text[:500]}")

## 3. Create Dataset and DataLoader

Notice what `create_lm_labels` does *not* do: mask anything. We mask pads via the attention mask in the training and evaluation loops (`targets[attention_mask[:, 1:] == 0] = -100`), so a genuine EOS stays supervised. GPT-2 pads with its EOS token (ID 50256): a value-based mask can't tell padding from a real end-of-text token and would silently delete every genuine EOS prediction, teaching the model to never stop.


In [ ]:
def create_lm_labels(token_ids):
    """
    Create shifted inputs and targets for language modeling.

    Args:
        token_ids: Token IDs (batch, seq)

    Returns:
        inputs: (batch, seq-1), all tokens except last
        targets: (batch, seq-1), all tokens except first
    """
    inputs = token_ids[:, :-1]    # Everything except last token
    targets = token_ids[:, 1:].clone()  # .clone(): the loops mark pads -100 in place

    # No masking here: pads are masked via the attention mask in the loops,
    # so a genuine EOS (GPT-2 pads with eos, ID 50256) stays supervised.
    return inputs, targets


from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    """Simple dataset that returns text chunks."""

    def __init__(self, text, chunk_size=512):
        """
        Args:
            text: The full text as a string
            chunk_size: How many characters per chunk
        """
        self.chunks = []

        # Keep the last complete chunk; discard only a shorter tail.
        for i in range(0, len(text) - chunk_size + 1, chunk_size):
            self.chunks.append(text[i:i + chunk_size])

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        return self.chunks[idx]


assert list(TextDataset("abcdefgh", chunk_size=4)) == ["abcd", "efgh"]
assert list(TextDataset("abcdefg", chunk_size=4)) == ["abcd"]


# Create dataset


def collate_fn(batch, tokenizer, max_length=128):
    """
    Tokenize and pad a batch of text strings.

    Args:
        batch: List of text strings
        tokenizer: The tokenizer to use
        max_length: Maximum sequence length

    Returns:
        input_ids: Padded token IDs (batch, max_length)
        attention_mask: 1 for real tokens, 0 for padding
    """
    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded["input_ids"], encoded["attention_mask"]


# Load tokenizer (GPT-2 has no pad token by default; reuse end-of-text)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# Create dataset: on TinyShakespeare, 512-character chunks average ~156
# GPT-2 tokens, so max_length=128 truncation drops the tail of most chunks.
# That's fine for pretraining: every chunk is still dense training signal
dataset = TextDataset(text, chunk_size=512)
print(f"Number of chunks: {len(dataset)}")

In [ ]:
# Create train/val split
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Create DataLoaders
collate = partial(collate_fn, tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# Check one batch
input_ids, attention_mask = next(iter(train_loader))
print(f"Batch input_ids shape: {input_ids.shape}")

## 4. Create Model

In [ ]:
# Small config for fast training
config = GPTConfig(
    vocab_size=50257,
    max_seq_len=128,
    embed_dim=256,
    num_heads=4,
    num_layers=4,
    d_ff=1024,
    dropout=0.1
)

model = MiniGPT(config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. The "Before" State: Untrained Model

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=1.0):
    """
    Generate text with temperature control.

    Temperature:
    - < 1.0: More focused/deterministic
    - = 1.0: Original distribution
    - > 1.0: More random/creative

    Args:
        model: Trained model
        tokenizer: Tokenizer
        prompt: Starting text
        max_new_tokens: How many tokens to generate
        temperature: Sampling temperature

    Returns:
        Generated text string
    """
    device = next(model.parameters()).device
    token_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    # do_sample=True: divide logits by temperature, then sample
    output_ids = model.generate(
        token_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(output_ids[0])


# Generate from untrained model
print("BEFORE TRAINING (random weights):")
print("="*50)
prompt = "The king"
print(f"Prompt: '{prompt}'")
print(f"Output: {generate(model, tokenizer, prompt)}")
print("\n(Random gibberish - the model hasn't learned anything yet!)")

## 6. Training Setup

In [ ]:
# Training hyperparameters
num_epochs = 3
learning_rate = 3e-4
warmup_steps = 100

total_steps = len(train_loader) * num_epochs
print(f"Total training steps: {total_steps}")

# Optimizer and scheduler
# Parameter-groups helper from Chapter 11
def get_parameter_groups(model, weight_decay=0.1):
    decay_params = []
    no_decay_params = []
    for name, param in model.named_parameters():
        if param.ndim < 2:
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    return [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': no_decay_params, 'weight_decay': 0.0}
    ]

# Parameter-groups helper from Chapter 11: matrices get weight decay,
# biases and LayerNorm vectors don't
optimizer = AdamW(get_parameter_groups(model, weight_decay=0.01),
                  lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

## 7. Training and Evaluation Functions

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device, clip_norm=1.0):
    """
    Train for one epoch.

    Returns:
        Average loss over the epoch
    """
    model.train()  # Enable dropout
    total_loss = 0

    progress = tqdm(dataloader, desc="Training")
    for input_ids, attention_mask in progress:
        # Move to device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        # Create inputs and targets (shifted by one), pads marked -100
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # ignore pads in the loss

        # ===== THE 5-STEP RECIPE =====

        # Step 1: Zero gradients
        optimizer.zero_grad(set_to_none=True)

        # Step 2: Forward pass
        logits = model(inputs)

        # Step 3: Compute loss
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape handles non-contiguous slices
            ignore_index=-100  # Skip padding positions
        )

        # Step 4: Backward pass
        loss.backward()

        # Important: Clip AFTER backward() but BEFORE step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

        # Step 5: Update weights
        optimizer.step()
        scheduler.step()

        # ===== END 5-STEP RECIPE =====

        total_loss += loss.item()

        # Update progress bar
        progress.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    """
    Evaluate the model on a dataset.

    Returns:
        avg_loss: Average cross-entropy loss
        perplexity: Exponentiated loss (interpretable metric)
    """
    model.eval()  # Disable dropout
    total_loss = 0
    total_tokens = 0

    for input_ids, attention_mask in dataloader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)

        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # ignore pads in the loss

        logits = model(inputs)

        # Compute loss
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)),
            targets.reshape(-1),  # reshape handles non-contiguous slices
            ignore_index=-100,
            reduction='sum'  # Sum instead of mean
        )

        # Count non-padding tokens only, so perplexity is honest
        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)

    return avg_loss, perplexity


def should_stop_early(val_losses, patience=3):
    """
    Check if we should stop training.

    Args:
        val_losses: List of validation losses per epoch
        patience: How many increasing epochs to tolerate

    Returns:
        True if we should stop
    """
    if len(val_losses) < patience + 1:
        return False

    # Check if last 'patience' losses are all increasing
    recent = val_losses[-(patience + 1):]
    for i in range(1, len(recent)):
        if recent[i] <= recent[i-1]:
            return False

    return True


In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, train_loss, val_loss, path):
    """
    Save training checkpoint, including RNG states for exact resume.

    Args:
        model: The model
        optimizer: The optimizer (has momentum state)
        scheduler: Learning rate scheduler
        epoch: Current epoch number
        train_loss: Training loss this epoch
        val_loss: Validation loss this epoch
        path: Where to save
    """
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        # RNG states: dropout and shuffling must continue the same sequence
        'rng_python': random.getstate(),
        'rng_numpy': np.random.get_state(),
        'rng_torch': torch.get_rng_state(),
        'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        'rng_mps': torch.mps.get_rng_state() if torch.backends.mps.is_available() else None,
    }, path)
    print(f"Checkpoint saved: {path}")


def load_checkpoint(model, optimizer, scheduler, path, device):
    """
    Load training checkpoint and restore RNG states.

    Returns:
        epoch: The epoch to resume from
        train_loss: Last training loss
        val_loss: Last validation loss
    """
    checkpoint = torch.load(path, map_location=device, weights_only=False)

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # RNG states must be CPU tensors: map_location=device can move them
    random.setstate(checkpoint['rng_python'])
    np.random.set_state(checkpoint['rng_numpy'])
    torch.set_rng_state(checkpoint['rng_torch'].cpu())
    if checkpoint['rng_cuda'] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint['rng_cuda']])
    if checkpoint['rng_mps'] is not None and torch.backends.mps.is_available():
        torch.mps.set_rng_state(checkpoint['rng_mps'].cpu())

    print(f"Checkpoint loaded from epoch {checkpoint['epoch']}")

    return checkpoint['epoch'], checkpoint['train_loss'], checkpoint['val_loss']


One trade-off to flag: `weights_only=True` refuses to execute arbitrary pickle code on load, the right habit for checkpoints from untrusted sources. But NumPy's RNG state contains NumPy scalars, which the safe loader rejects, so this resume-exact version uses `weights_only=False`. Only load checkpoints you created yourself (or explicitly trust) this way; for anything downloaded, prefer `weights_only=True` and accept that the RNG states won't restore. Two more conditions for a bit-exact resume: rebuild the scheduler with the same total-step horizon as the original run, and consume batches in the same order.

## 8. Training Loop

In [ ]:
# Training!
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"{'='*50}")

    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    train_losses.append(train_loss)

    # Evaluate
    val_loss, perplexity = evaluate(model, val_loader, device)
    val_losses.append(val_loss)

    print(f"\nTrain Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Perplexity: {perplexity:.1f}")

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, scheduler, epoch,
                       train_loss, val_loss, "best_model.pt")

print("\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

## 9. Plot Loss Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
epochs = range(1, len(train_losses) + 1)
plt.plot(epochs, train_losses, 'b-o', label='Train Loss')
plt.plot(epochs, val_losses, 'r-o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. The Payoff: Text Generation!

In [ ]:
# Load best model (our own checkpoint, so weights_only=False is safe here;
# the RNG states inside need the full loader, see the checkpoint section)
checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded checkpoint from epoch {checkpoint['epoch'] + 1}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}")

In [ ]:
print("\n" + "="*60)
print("AFTER TRAINING (on Shakespeare):")
print("="*60)

prompts = [
    "The king",
    "To be or not to be",
    "Friends, Romans, countrymen",
    "All the world's a stage"
]

for prompt in prompts:
    output = generate(model, tokenizer, prompt, max_new_tokens=40, temperature=0.8)
    print(f"\nPrompt: '{prompt}'")
    print(f"Output: {output}")
    print("-"*40)

## 11. Before vs After Comparison

In [ ]:
# Create fresh untrained model for comparison
set_seed(123)  # Different seed for different random weights
untrained_model = MiniGPT(config).to(device)
untrained_model.eval()

prompt = "The fair maiden"

print("="*60)
print(f"Prompt: '{prompt}'")
print("="*60)
print("\nBEFORE (untrained):")
print(generate(untrained_model, tokenizer, prompt, max_new_tokens=30))
print("\nAFTER (trained on Shakespeare):")
print(generate(model, tokenizer, prompt, max_new_tokens=30, temperature=0.8))
print("\nSame architecture. Same code. Training makes all the difference!")

## 12. Temperature Comparison

In [ ]:
prompt = "The noble lord"

print(f"Prompt: '{prompt}'\n")
for temp in [0.5, 0.8, 1.0, 1.5]:
    output = generate(model, tokenizer, prompt, max_new_tokens=30, temperature=temp)
    print(f"Temperature {temp}:")
    print(f"  {output}")
    print()

## Summary

**What we built:**

1. **Label shifting** for next-token prediction
2. **DataLoaders** that efficiently batch and tokenize text
3. **The 5-step training recipe**: zero_grad → forward → loss → backward → step
4. **Evaluation** with validation loss and perplexity
5. **Checkpointing** to save and resume training
6. **Text generation** with temperature control

**Key concepts:**

- Training is a feedback control loop: measure error, adjust weights, repeat
- Cross-entropy loss penalizes confident wrong predictions more than uncertain ones
- Overfitting = memorizing training data (train loss ↓, val loss ↑)
- Perplexity measures "how surprised" the model is—lower is better
- Temperature controls generation diversity

**Next:** Chapter 13 will teach you to fine-tune this model for specific tasks!

## Exercises

These three notebook experiments have their own numbering. For Chapter 12's garden corpus and document split, use **book Exercise 1**; for checkpoint resume after a restart, use **book Exercise 4**. The learning-rate experiment below is **book Exercise 2**.

### Exercise 1: Learning Rate Experiment

Try different learning rates and compare results.

In [ ]:
# YOUR CODE HERE
# Train with lr=1e-5, lr=3e-4, lr=1e-2
# Compare the loss curves
# Which learning rate works best?

### Exercise 2: More Epochs

In [ ]:
# YOUR CODE HERE
# Train for 5-10 epochs instead of 3
# Does the model keep improving?
# Do you see signs of overfitting?

### Exercise 3: Model Size Comparison

In [ ]:
# YOUR CODE HERE
# Create a smaller model (2 layers, 128 dim)
# Create a larger model (6 layers, 384 dim)
# Compare training speed and final perplexity